In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv('PATH_OF_TELEMETRY_FILE_CSV')

df.shape
df.dtypes
df.isna().sum()
df.duplicated().sum()

df.describe()
df.columns

df.groupby('state')[['temperature', 'battery_voltage', 'signal_strength',
                     'current_draw', 'rotation_rate', 'packet_loss',
                     'solar_current']].mean()


# Feature engineering

df['current_voltage_relation'] = df['current_draw'] / df['battery_voltage'].replace(0, np.nan)
df['current_solar_relation'] = df['solar_current'] - df['current_draw']
df['communication_stress'] = (-df['signal_strength']) * df['packet_loss']

temp_mean = df['temperature'].mean()
df['temp_deviation'] = abs(df['temperature'] - temp_mean)

print(df[['current_voltage_relation', 'current_solar_relation',
          'communication_stress', 'temp_deviation']].isna().sum())

print(np.isinf(df[['current_voltage_relation', 'current_solar_relation',
                   'communication_stress', 'temp_deviation']]).sum())


# Features and target

features = [
    'temperature',
    'battery_voltage',
    'signal_strength',
    'current_draw',
    'rotation_rate',
    'packet_loss',
    'solar_current',
    'current_voltage_relation',
    'current_solar_relation',
    'communication_stress',
    'temp_deviation'
]

X = df[features]
y = df['state']


# Label encoding

encoder = LabelEncoder()
y = encoder.fit_transform(y)

print(encoder.classes_)


# Train / validation / test split

x_train, x_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


# Fill missing values using training data only

train_medians = x_train.median()

x_train = x_train.fillna(train_medians)
x_val = x_val.fillna(train_medians)
x_test = x_test.fillna(train_medians)


# Check missing values

print("X train NaN:", x_train.isna().sum().sum())
print("X val NaN:", x_val.isna().sum().sum())
print("X test NaN:", x_test.isna().sum().sum())

print("X train Inf:", np.isinf(x_train).sum().sum())
print("X val Inf:", np.isinf(x_val).sum().sum())
print("X test Inf:", np.isinf(x_test).sum().sum())


# Scaling

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)


# Convert to tensors

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

x_train = torch.tensor(x_train, dtype=torch.float32)
x_val = torch.tensor(x_val, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.int64)
y_val = torch.tensor(y_val, dtype=torch.int64)
y_test = torch.tensor(y_test, dtype=torch.int64)


# Datasets

train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
test_dataset = TensorDataset(x_test, y_test)


# DataLoaders

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


# Model

class Telemetry(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(11, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 4)
        )

    def forward(self, x):
        return self.model(x)


# Trainer

class TelemetryTrainer:
    def __init__(self, model, optimizer, criterion):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion

    def train_one_epoch(self, train_loader):
        self.model.train()
        train_loss = 0

        for x_batch, y_batch in train_loader:
            output = self.model(x_batch)
            loss = self.criterion(output, y_batch)

            self.optimizer.zero_grad(set_to_none=True)
            loss.backward()
            self.optimizer.step()

            train_loss += loss.item()

        return train_loss / len(train_loader)

    def validate(self, val_loader):
        self.model.eval()

        total_loss = 0
        correct = 0
        total = 0

        with torch.inference_mode():
            for x_batch, y_batch in val_loader:
                output = self.model(x_batch)
                loss = self.criterion(output, y_batch)

                total_loss += loss.item()

                predictions = output.argmax(dim=1)

                correct += (predictions == y_batch).sum().item()
                total += y_batch.size(0)

        avg_loss = total_loss / len(val_loader)
        accuracy = correct / total

        return avg_loss, accuracy

    def fit(self, train_loader, val_loader, epochs):
        for epoch in range(epochs):
            train_loss = self.train_one_epoch(train_loader)
            val_loss, val_acc = self.validate(val_loader)

            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2%}")


model = Telemetry()

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

trainer = TelemetryTrainer(model, optimizer, criterion)

trainer.fit(train_loader, val_loader, epochs=500)


# Final evaluation

val_loss, val_acc = trainer.validate(val_loader)
test_loss, test_acc = trainer.validate(test_loader)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.2%}")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2%}")

current_voltage_relation    3
current_solar_relation      3
communication_stress        3
temp_deviation              1
dtype: int64
current_voltage_relation    0
current_solar_relation      0
communication_stress        0
temp_deviation              0
dtype: int64
['COMMUNICATION_ISSUE' 'NORMAL' 'POWER_ISSUE' 'SENSOR_ANOMALY']
X train NaN: 0
X val NaN: 0
X test NaN: 0
X train Inf: 0
X val Inf: 0
X test Inf: 0
Epoch 100/500 - Train Loss: 0.0021 - Val Loss: 0.0018 - Val Acc: 100.00%
Epoch 200/500 - Train Loss: 0.0005 - Val Loss: 0.0007 - Val Acc: 100.00%
Epoch 300/500 - Train Loss: 0.0002 - Val Loss: 0.0004 - Val Acc: 100.00%
Epoch 400/500 - Train Loss: 0.0001 - Val Loss: 0.0003 - Val Acc: 100.00%
Epoch 500/500 - Train Loss: 0.0001 - Val Loss: 0.0003 - Val Acc: 100.00%
Validation Loss: 0.0003
Validation Accuracy: 100.00%
Test Loss: 0.0010
Test Accuracy: 100.00%
